# Concept Score Vector Generation

Use ConceptCLIP's text encoder to run inference on concept groups at four granularities (30/50/70/90),
compute the mean text embedding across all prompt templates (concept mean embedding),
then perform matrix multiplication with existing image embeddings to produce concept score vectors.

**Input:** `data/image_features_{train,val,test}.csv` (generated by `[1]image_embedding_inference.ipynb`)

**Output (per group):**
- `data/concept_{N}/text_embeddings-concepts{N}_mean.csv`
- `data/concept_{N}/concept{N}_logits_{train,val,test}.csv`

# 1 Define Concepts and Prompt Templates

In [ ]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# ── Full 90-concept metadata (level 2→30, 3→50, 4→70, 5→90) ──
cell_feature_metadata = {
    "nuclear_morphology_nc": {
        "en": "Nuclear morphology and N:C ratio",
        "zh": "细胞核形态与核浆结构",
        "features": [
            {"en": "Segmented nucleus", "zh": "分叶细胞核", "level": 2},
            {"en": "Band nucleus (band form)", "zh": "带状细胞核（未分叶）", "level": 2},
            {"en": "Bilobed nucleus", "zh": "双叶细胞核", "level": 2},
            {"en": "Round nucleus", "zh": "圆形细胞核", "level": 2},
            {"en": "Reniform / indented nucleus", "zh": "肾形/凹陷细胞核", "level": 2},
            {"en": "Coarse chromatin pattern", "zh": "染色质粗糙/团块状", "level": 2},
            {"en": "Prominent nucleoli", "zh": "明显核仁", "level": 2},
            {"en": "Irregular nuclear membrane", "zh": "不规则核膜", "level": 2},
            {"en": "Eccentric nucleus", "zh": "偏心细胞核", "level": 3},
            {"en": "Binucleated cell", "zh": "双核细胞", "level": 3},
            {"en": "Multinucleated cell", "zh": "多核细胞", "level": 3},
            {"en": "Clefted / notched nucleus", "zh": "裂隙/切迹样细胞核", "level": 3},
            {"en": "Oval nucleus", "zh": "椭圆形细胞核", "level": 3},
            {"en": "Pyknotic nucleus", "zh": "固缩细胞核", "level": 3},
            {"en": "Apoptotic nuclear fragments", "zh": "凋亡样核碎片", "level": 3},
            {"en": "Hypersegmented nucleus", "zh": "过度分叶细胞核", "level": 4},
            {"en": "Fine/open chromatin pattern", "zh": "染色质细致/疏松", "level": 4},
            {"en": "High nuclear-to-cytoplasmic ratio", "zh": "高核浆比", "level": 4},
            {"en": "Nuclear blebs / micronuclei", "zh": "核芽/微核", "level": 4},
            {"en": "Marked nuclear size variation (anisokaryosis)", "zh": "明显核大小不一（核异大）", "level": 4},
            {"en": "Convoluted / folded nucleus", "zh": "皱褶/卷曲细胞核", "level": 5},
            {"en": "Peripheral chromatin condensation (chromatin margination)", "zh": "核周染色质凝集", "level": 5},
        ],
    },
    "cytoplasmic_tone_texture": {
        "en": "Cytoplasmic tone and texture",
        "zh": "胞质色调与质地",
        "features": [
            {"en": "Basophilic cytoplasm", "zh": "嗜碱性胞质", "level": 2},
            {"en": "Pale cytoplasm", "zh": "淡染胞质", "level": 2},
            {"en": "Cytoplasmic vacuoles", "zh": "胞质空泡", "level": 2},
            {"en": "Ground-glass cytoplasm", "zh": "毛玻璃样胞质", "level": 2},
            {"en": "Foamy cytoplasm", "zh": "泡沫样胞质", "level": 2},
            {"en": "Cytoplasmic blebs / projections", "zh": "胞质突起/胞质芽", "level": 2},
            {"en": "Perinuclear hof (Golgi zone clearing)", "zh": "核周晕（高尔基区透亮带）", "level": 2},
            {"en": "Polarized cytoplasm (one-sided)", "zh": "胞质极化（单侧胞质集中）", "level": 3},
            {"en": "Homogeneous dense cytoplasm", "zh": "均质致密胞质", "level": 3},
            {"en": "Paranuclear cytoplasmic vacuoles", "zh": "核周胞质空泡", "level": 3},
            {"en": "Clear cytoplasm", "zh": "透明样胞质", "level": 4},
            {"en": "Plasmacytoid cytoplasm", "zh": "浆细胞样胞质", "level": 4},
            {"en": "Peripheral basophilic rim", "zh": "细胞周边嗜碱性环", "level": 4},
            {"en": "Cytoplasmic streaming / tails", "zh": "胞质拖尾/流动样改变", "level": 5},
            {"en": "Reticular / lace-like cytoplasm", "zh": "网状/蕾丝样胞质纹理", "level": 5},
            {"en": "Patchy basophilic cytoplasm", "zh": "片状嗜碱性胞质区域", "level": 5},
        ],
    },
    "cytoplasmic_granules": {
        "en": "Cytoplasmic granules and inclusions",
        "zh": "胞质颗粒与内含物",
        "features": [
            {"en": "Fine azurophilic granules", "zh": "细嗜天青颗粒", "level": 2},
            {"en": "Eosinophilic granules", "zh": "嗜酸性颗粒（橙红）", "level": 2},
            {"en": "Basophilic granules", "zh": "嗜碱性颗粒（深紫）", "level": 2},
            {"en": "Toxic granulation", "zh": "中毒性颗粒", "level": 2},
            {"en": "Dohle bodies", "zh": "Döhle小体", "level": 2},
            {"en": "Auer rods", "zh": "Auer小体", "level": 2},
            {"en": "Granules obscure nucleus", "zh": "颗粒掩盖细胞核", "level": 2},
            {"en": "Hypogranular / agranular cytoplasm", "zh": "颗粒减少/无颗粒胞质", "level": 2},
            {"en": "Coarse cytoplasmic granules", "zh": "粗大胞质颗粒", "level": 3},
            {"en": "Giant cytoplasmic granules", "zh": "巨型胞质颗粒", "level": 3},
            {"en": "Azurophilic granules in lymphocytes", "zh": "淋巴细胞嗜天青颗粒", "level": 3},
            {"en": "Cytoplasmic crystalline inclusions", "zh": "晶样胞质内含物", "level": 3},
            {"en": "Cytoplasmic pigment granules", "zh": "色素颗粒（如含铁颗粒）", "level": 3},
            {"en": "Phagocytosed cellular debris", "zh": "吞噬细胞碎片/红细胞", "level": 3},
            {"en": "Basophilic stippling of erythrocytes", "zh": "红细胞嗜碱性点彩", "level": 4},
            {"en": "Howell\u2013Jolly bodies in erythrocytes", "zh": "红细胞内 Howell\u2013Jolly 小体", "level": 4},
            {"en": "Pappenheimer bodies in erythrocytes", "zh": "红细胞内 Pappenheimer 小体", "level": 4},
            {"en": "Uneven granule size (anisogranularity)", "zh": "颗粒大小不均（颗粒异大）", "level": 4},
            {"en": "Polar aggregation of cytoplasmic granules", "zh": "胞质颗粒向一极聚集", "level": 4},
            {"en": "Perinuclear sparing of cytoplasmic granules", "zh": "核周颗粒缺如带", "level": 4},
            {"en": "Uniform fine cytoplasmic granules", "zh": "均匀细小胞质颗粒", "level": 5},
            {"en": "Peripheral rim of cytoplasmic granules", "zh": "细胞周边颗粒环状分布", "level": 5},
            {"en": "Perinuclear ring of cytoplasmic granules", "zh": "核周颗粒环状分布", "level": 5},
            {"en": "Discrete non-granular cytoplasmic inclusions", "zh": "离散非颗粒性胞质内含物", "level": 5},
            {"en": "Single large cytoplasmic inclusion", "zh": "单个大型胞质内含物", "level": 5},
        ],
    },
    "non_leukocyte_elements": {
        "en": "Non-leukocyte blood elements",
        "zh": "非白细胞血细胞要素",
        "features": [
            {"en": "Nucleated erythrocyte (erythroblast)", "zh": "有核红细胞（幼红细胞）", "level": 2},
            {"en": "Platelet fragments / clumps", "zh": "血小板碎片/成团", "level": 2},
            {"en": "Red blood cell central pallor", "zh": "红细胞中央淡染区", "level": 2},
            {"en": "Red blood cell fragments (schistocytes)", "zh": "红细胞碎片（裂红细胞）", "level": 3},
            {"en": "Spherocytic red blood cell", "zh": "球形红细胞（无中央淡染）", "level": 3},
            {"en": "Target red blood cell (codocyte)", "zh": "靶形红细胞", "level": 3},
            {"en": "Teardrop red blood cell (dacrocyte)", "zh": "泪滴形红细胞", "level": 3},
            {"en": "Red blood cell anisocytosis (size variation)", "zh": "红细胞大小不一（红细胞异大）", "level": 4},
            {"en": "Red blood cell poikilocytosis (shape variation)", "zh": "红细胞形态多样（红细胞异形）", "level": 4},
            {"en": "Microcytic red blood cells", "zh": "小红细胞", "level": 4},
            {"en": "Macrocytic red blood cells", "zh": "大红细胞", "level": 4},
            {"en": "Rouleaux formation of red blood cells", "zh": "红细胞缗钱状排列", "level": 4},
            {"en": "Giant platelets", "zh": "巨大血小板", "level": 4},
            {"en": "Oval / elliptic red blood cell (ovalocyte / elliptocyte)", "zh": "卵圆形/椭圆形红细胞（卵圆红细胞）", "level": 5},
            {"en": "Stomatocytic red blood cell (stomatocyte)", "zh": "口形红细胞（裂口样中央淡染）", "level": 5},
            {"en": "Echinocytic red blood cell (burr cell)", "zh": "棘状红细胞（burr cell）", "level": 5},
            {"en": "Acanthocytic red blood cell (spur cell)", "zh": "棘突红细胞（spur cell）", "level": 5},
            {"en": "Hypochromic red blood cell", "zh": "低色素红细胞", "level": 5},
            {"en": "Polychromatophilic red blood cell", "zh": "多染性红细胞", "level": 5},
            {"en": "Platelet anisocytosis (size variation)", "zh": "血小板大小不一（血小板异大）", "level": 5},
            {"en": "Hypogranular platelets", "zh": "低颗粒血小板", "level": 5},
            {"en": "Hypergranular platelets", "zh": "高颗粒血小板", "level": 5},
            {"en": "Platelet satellitism around leukocytes", "zh": "血小板围绕白细胞卫星状分布", "level": 5},
        ],
    },
    "artifacts_quality": {
        "en": "Preparation and technical artifacts",
        "zh": "制片与技术伪影",
        "features": [
            {"en": "Stain precipitate (artifact)", "zh": "染色沉淀（伪影）", "level": 2},
            {"en": "Overlapping cell clumps (artifact)", "zh": "细胞重叠/成团（伪影）", "level": 2},
            {"en": "Out-of-focus (artifact)", "zh": "失焦（伪影）", "level": 2},
            {"en": "Motion blur (artifact)", "zh": "运动模糊（伪影）", "level": 2},
        ],
    },
}


def get_cell_features(metadata: dict, level: int = 1):
    """Filter by level, keeping all features with feature['level'] <= level."""
    cell_features = []
    for category in metadata.values():
        for feat in category.get('features', []):
            if feat.get('level', 1) <= level:
                cell_features.append(feat)
    return cell_features


# ── Prompt templates (shared across all four granularities) ──
PROMPT_TEMPLATES = [
    "a blood cell photo with sign of {}",
    "a photo of a blood cell with {}",
    "a blood cell image indicating {}",
    "an image of a blood cell showing {}",
    "peripheral blood cell with {}",
    "a peripheral blood cell photo with sign of {}",
    "a photo of a blood cell with {}",
    "a peripheral blood cell image indicating {}",
    "an image of blood cell showing {}",
]

# ── Granularity config: level → (concept count, output directory) ──
GRANULARITY_CONFIGS = [
    {"level": 2, "n": 30, "dir": "concept_30"},
    {"level": 3, "n": 50, "dir": "concept_50"},
    {"level": 4, "n": 70, "dir": "concept_70"},
    {"level": 5, "n": 90, "dir": "concept_90"},
]

IMG_FEAT_DIR = Path('./data')  # directory containing image_features CSVs

# Validate concept count per group
for cfg in GRANULARITY_CONFIGS:
    feats = get_cell_features(cell_feature_metadata, level=cfg['level'])
    print(f"level={cfg['level']}  expected={cfg['n']}  actual={len(feats)}  ok={len(feats)==cfg['n']}")

# 2 Load ConceptCLIP Model

In [ ]:
from transformers import AutoModel, AutoProcessor
from huggingface_hub import login, whoami
import os

# Set the HF_TOKEN environment variable before running:
#   Windows: set HF_TOKEN=<your_token>
#   Linux/Mac: export HF_TOKEN=<your_token>
# Or run interactively: huggingface-cli login
hf_token = os.environ.get("HF_TOKEN")
login(token=hf_token)
print(whoami())

In [ ]:
model     = AutoModel.from_pretrained('JerrryNie/ConceptCLIP', trust_remote_code=True)
processor = AutoProcessor.from_pretrained('JerrryNie/ConceptCLIP', trust_remote_code=True)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model  = model.to(device).eval()
print('Device:', device)

# Extract model logit scale parameter
def _get_scale(model):
    for attr in ['concept_logit_scale', 'logit_scale']:
        if hasattr(model, attr):
            v = getattr(model, attr)
            if torch.is_tensor(v):
                return float(v.detach().item())
            return float(v)
    return 1.0

SCALE = _get_scale(model)
print('logit scale:', SCALE)

# 3 Generate Concept Mean Embeddings and Score Vectors

For each granularity:
1. Combine each concept × prompt template and encode with the text encoder
2. Average across templates → concept mean embedding → save as CSV
3. Load image_features CSV, compute `scale × (norm_img @ norm_text.T)` → concept logits → save as CSV

In [ ]:
def encode_concepts_mean(concept_names, templates, model, processor, device):
    """Encode all template prompts for each concept and average the CLS embeddings. Returns [K, D] float32."""
    K = len(concept_names)
    accum = None
    with torch.no_grad():
        for tmpl in templates:
            texts = [tmpl.format(c) for c in concept_names]
            inputs = processor(text=texts, return_tensors='pt', padding=True, truncation=True).to(device)
            text_cls, _ = model.encode_text(inputs['input_ids'], normalize=True)  # [K, D]
            text_cls = text_cls.float().cpu()
            if accum is None:
                accum = torch.zeros_like(text_cls)
            accum += text_cls
    mean_emb = (accum / len(templates)).numpy()  # [K, D]
    return mean_emb


def save_mean_embedding_csv(embeddings, concept_names, out_path):
    """Save concept mean embeddings as a CSV file (columns: name, f0, f1, ...)."""
    K, D = embeddings.shape
    cols = ['name'] + [f'f{i}' for i in range(D)]
    df = pd.DataFrame(
        np.concatenate([np.array(concept_names, dtype=object).reshape(-1, 1), embeddings], axis=1),
        columns=cols,
    )
    for c in cols[1:]:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f'  Saved mean embedding: {out_path}  shape=({K}, {D})')


def compute_and_save_logits(concept_emb, concept_names, scale, img_feat_dir, out_dir, prefix):
    """Load image features CSV, compute concept logits, and save to CSV."""
    # L2-normalize concept embeddings
    c_norm = np.linalg.norm(concept_emb, axis=1, keepdims=True) + 1e-8
    concept_normed = concept_emb / c_norm

    for split in ['train', 'val', 'test']:
        img_csv = img_feat_dir / f'image_features_{split}.csv'
        if not img_csv.exists():
            print(f'  [WARN] {img_csv} not found, skip')
            continue
        df = pd.read_csv(img_csv)
        img_feats = df.iloc[:, 3:].to_numpy(dtype=np.float32)  # skip id, label, split columns
        # L2-normalize image features
        i_norm = np.linalg.norm(img_feats, axis=1, keepdims=True) + 1e-8
        img_normed = img_feats / i_norm
        # cosine similarity × scale
        logits = scale * (img_normed @ concept_normed.T)  # [N, K]
        # Build output DataFrame
        out_df = df[['id', 'label', 'split']].copy()
        for i, cname in enumerate(concept_names):
            out_df[cname.replace(',', ' ')] = logits[:, i]
        out_path = out_dir / f'{prefix}_logits_{split}.csv'
        out_df.to_csv(out_path, index=False, encoding='utf-8-sig')
        print(f'  Saved logits: {out_path}  shape={out_df.shape}')


# ── Main loop: process each granularity ──
for cfg in GRANULARITY_CONFIGS:
    n = cfg['n']
    out_dir = IMG_FEAT_DIR / cfg['dir']
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f'\n=== Concept {n} (level<={cfg["level"]}) ===')

    # 1) Get concept list
    feats = get_cell_features(cell_feature_metadata, level=cfg['level'])
    concept_names = [f['en'] for f in feats]

    # 2) Encode and average
    mean_emb = encode_concepts_mean(concept_names, PROMPT_TEMPLATES, model, processor, device)

    # 3) Save mean embedding CSV
    save_mean_embedding_csv(mean_emb, concept_names,
                           out_dir / f'text_embeddings-concepts{n}_mean.csv')

    # 4) Compute and save logits CSV
    compute_and_save_logits(mean_emb, concept_names, SCALE,
                           IMG_FEAT_DIR, out_dir, prefix=f'concept{n}')

print('\nDone.')

# 4 Quick Validation

In [ ]:
for cfg in GRANULARITY_CONFIGS:
    n = cfg['n']
    d = IMG_FEAT_DIR / cfg['dir']
    print(f'--- concept_{n} ---')
    for f in sorted(d.glob('*.csv')):
        df = pd.read_csv(f, nrows=2)
        print(f'  {f.name}  cols={len(df.columns)}  preview: {df.columns[:5].tolist()}')